# 54 — Resume Ranking
**Goal:** Rank multiple resumes against a job description for candidate shortlisting.

Individual scoring answers "is this candidate viable?"; **ranking** answers "which candidates first?" `ResumeRanker` loops over a batch of `(name, text)` resumes, scores each against one JD, sorts descending, and labels each with a shortlist status — the classic recruiter triage view.

**Why it matters for resumes / ATS:** a hiring pipeline sees hundreds of resumes per role. Ranking turns the scoring machinery into a single sortable list with a defensible threshold (≥80 SHORTLIST, ≥60 MAYBE, else PASS), so recruiters can work from the top down and candidates can see exactly what separated first from last.

## 1. Batch Resume Ranking

`rank(resumes, jd_text)` scores every resume in the batch, sorts by score descending, and returns a list of dicts — `name`, `score`, `skills_found`.

**What the code does:**
- `__init__` tries to attach a `ResumeJDMatcher` (a semantic matcher from later chapters) but checks `'ResumeJDMatcher' in dir()` — and inside a method `dir()` sees only **local** names, so that is never true here and the simplified scorer always runs. The hook is ready for a real matcher imported at module level.
- The simplified path reuses a fixed 6-skill vocabulary: `skill_ratio = matched JD skills / total JD skills`, then `score = ratio * 80 + 10`. The +10 floor guarantees even a zero-match resume scores 10; the ×80 cap means the top score is 90, leaving headroom for a future embedding term (`details["embedding"]` is hard-coded to `0.5` as a placeholder).
- `skills_found` counts the resume's hits against that same vocabulary.
- Sorting is `reverse=True` on score; Python's sort is stable, so ties keep input order.

**Expected (verified by running):** against the sample JD the output is `#1 Alice 90.0`, `#2 Diana 63.3`, `#3 Charlie 36.7`, `#4 Bob 10.0`. Alice matches all three JD skills (Python, NLP, TensorFlow); Diana matches two; Charlie's three hits (Python, SQL, AWS) overlap the JD by only Python; Bob (Java/Spring) matches none and gets the 10-point floor. `skills_found` counts vocabulary hits, not JD hits — Charlie shows 3 yet ranks below Diana's 2.

In [ ]:
class ResumeRanker:
    def __init__(self):
        self.matcher = ResumeJDMatcher() if 'ResumeJDMatcher' in dir() else None
    
    def rank(self, resumes, jd_text):
        results = []
        for i, (name, text) in enumerate(resumes):
            if self.matcher:
                match = self.matcher.match(text, jd_text)
                score = match['score']
                details = match['details']
            else:
                # Simplified scoring
                skills = ["Python", "NLP", "TensorFlow", "SQL", "AWS", "Docker"]
                jd_skills = [s for s in skills if s.lower() in jd_text.lower()]
                resume_skills = [s for s in skills if s.lower() in text.lower()]
                skill_ratio = len([s for s in jd_skills if s in resume_skills]) / max(len(jd_skills), 1)
                score = skill_ratio * 80 + 10
                details = {"skill_match": skill_ratio, "embedding": 0.5}
            
            results.append({
                "name": name, "score": round(score * 100, 1) if score <= 1 else round(score, 1),
                "skills_found": len(resume_skills) if 'resume_skills' in dir() else 0,
            })
        
        results.sort(key=lambda x: x["score"], reverse=True)
        return results

resumes = [
    ("Alice", "Senior data scientist, Python, NLP, TensorFlow, 5 years"),
    ("Bob", "Java backend developer, Spring Boot, microservices, 3 years"),
    ("Charlie", "Data engineer, Python, SQL, Spark, AWS, 4 years"),
    ("Diana", "NLP researcher, Python, PyTorch, Transformers, BERT, PhD"),
]
jd = "Senior Data Scientist: Python, NLP, TensorFlow required, 5+ years"

ranker = ResumeRanker()
ranked = ranker.rank(resumes, jd)
print(f"Ranking for: {jd}")
print("=" * 50)
for i, r in enumerate(ranked):
    print(f"  #{i+1} {r['name']:10s} | Score: {r['score']:.1f} | Skills: {r['skills_found']}")

## 2. Display Results

Ranking is only useful if a human can scan it — this cell renders the sorted list as a **status table** with recruiter-style thresholds.

**What the code does:** prints a header row, then for each ranked candidate a `Rank`/`Name`/`Score`/`Status` row, where status is assigned by band: `score >= 80 → SHORTLIST`, `>= 60 → MAYBE`, else `PASS`. The thresholds are deliberately simple — a transparent cut rule the whole team can argue about, exactly like the Ch. 50 weights.

**Expected (verified by running):** Alice (90.0) is `SHORTLIST`, Diana (63.3) is `MAYBE`, Charlie (36.7) and Bob (10.0) are `PASS`. The cutoff logic runs *after* sorting, so the table reads top-down as a funnel: two candidates worth a call, two parked.

In [ ]:
# Display as table
print(f"\\n{'Rank':<6}{'Name':<12}{'Score':<10}{'Status':<12}")
print("-" * 40)
for i, r in enumerate(ranked):
    if r['score'] >= 80:
        status = "SHORTLIST"
    elif r['score'] >= 60:
        status = "MAYBE"
    else:
        status = "PASS"
    print(f"{'#'+str(i+1):<6}{r['name']:<12}{r['score']:<10}{status:<12}")

## Summary: Resume ranking aggregates all ATS dimensions into a single sortable score.

**Ranking is the payoff of everything before it — one sortable score per candidate, with thresholds anyone can read.**

`ResumeRanker` batches Ch. 50's rules, Ch. 51's explainability, and Ch. 52's pipeline into a single descending list, and the 80/60 bands turn that list into a triage funnel (SHORTLIST / MAYBE / PASS). The simplified scorer is honest about its limits: vocabulary-only matching, a 90-point ceiling, and an `embedding: 0.5` placeholder waiting for real semantics.

That placeholder is the bridge to what comes next: Ch. 55 (OpenRouter Setup) introduces the LLM layer that will replace hard-coded skill lists and heuristic scores with semantic, generative matching.